# Training TBModel on Auditory Cortex Data with CWN (CW Network)

This notebook demonstrates **graph-level classification** on the A123 mouse auditory cortex dataset
using a **CWN (CW Network)** backbone from [TopoModelX](https://github.com/pyt-team/TopoModelX).

CWN operates on **cell complexes** (rank 0 = nodes, rank 1 = edges, rank 2 = 2-cells from cycles).
We use the `CellCycleLifting` transform to lift graphs into cell complexes, then train
with TopoBench's `TBModel`, `CWNWrapper`, and `PropagateSignalDown` readout.

**Task**: Predict frequency bin (0–8) from graph structure.

Requirements: the project installed in PYTHONPATH and optional dependencies (torch_geometric, topomodelx, toponetx).

In [ ]:
import os
os.chdir('..')

In [ ]:
import torch
import numpy as np
import lightning as pl
from functools import partial
from omegaconf import OmegaConf

from topobench.data.loaders.graph.a123_loader import A123DatasetLoader
from topobench.dataloader.dataloader import TBDataloader
from topobench.data.preprocessor import PreProcessor

from topobench.model.model import TBModel
from topomodelx.nn.cell.cwn import CWN
from topobench.nn.wrappers.cell.cwn_wrapper import CWNWrapper
from topobench.nn.encoders import AllCellFeatureEncoder
from topobench.nn.readouts import PropagateSignalDown

from topobench.loss.loss import TBLoss
from topobench.optimizer import TBOptimizer
from topobench.evaluator.evaluator import TBEvaluator

print('Imports OK')

In [ ]:
# Configuration for graph-level classification
# CWN operates on cell complexes, so we need graph data with edge structure

loader_config = OmegaConf.create({
    'data_domain': 'graph',
    'data_type': 'A123',
    'data_name': 'a123_cortex_m',
    'data_dir': './data/a123/',
    'corr_threshold': 0.3,
    'specific_task': 'classification',
})

# CellCycleLifting: finds cycles in the graph and lifts them to 2-cells
# ProjectionSum creates higher-rank features via incidence matrix projection
transform_config = OmegaConf.create({
    'transform_type': 'lifting',
    'transform_name': 'CellCycleLifting',
    'complex_dim': 3,
    'max_cell_length': 18,
    'feature_lifting': 'ProjectionSum',
    'preserve_edge_attr': False,
})

split_config = OmegaConf.create({
    'learning_setting': 'inductive',
    'split_type': 'random',
    'data_seed': 0,
    'data_split_dir': './data/a123/splits/',
    'train_prop': 0.5,
})

dim_hidden = 16
in_channels = 3   # node features: mean_corr, std_corr, noise_diag
out_channels = 9   # frequency bins 0-8
n_cell_dims = 3    # ranks 0, 1, 2 for the cell complex

readout_config = OmegaConf.create({
    'readout_name': 'PropagateSignalDown',
    'num_cell_dimensions': n_cell_dims,
    'hidden_dim': dim_hidden,
    'out_channels': out_channels,
    'task_level': 'graph',
    'pooling_type': 'sum',
})

loss_config = OmegaConf.create({
    'dataset_loss': {
        'task': 'classification',
        'loss_type': 'cross_entropy',
    }
})

evaluator_config = OmegaConf.create({
    'task': 'classification',
    'num_classes': out_channels,
    'metrics': ['f1', 'precision', 'recall', 'accuracy'],
})

optimizer_config = OmegaConf.create({
    'optimizer_id': 'Adam',
    'parameters': {'lr': 0.001, 'weight_decay': 0.0005},
})

print('Configs created')
print(f'Hidden dim: {dim_hidden}, In channels: {in_channels}, Out channels: {out_channels}')
print(f'Cell complex dimensions: {n_cell_dims} (ranks 0, 1, 2)')

In [ ]:
# Load the A123 dataset and apply CellCycleLifting

graph_loader = A123DatasetLoader(loader_config)
dataset, dataset_dir = graph_loader.load()
print(f'Dataset loaded: {len(dataset)} samples')

# Apply CellCycleLifting to create cell complex structure
# This produces x_0 (nodes), x_1 (edges), x_2 (2-cells from cycles)
# along with incidence and adjacency matrices needed by CWN
preprocessor = PreProcessor(dataset, dataset_dir, transform_config)
dataset_train, dataset_val, dataset_test = preprocessor.load_dataset_splits(split_config)

print(f'Dataset splits created:')
print(f'  Train: {len(dataset_train)} samples')
print(f'  Val:   {len(dataset_val)} samples')
print(f'  Test:  {len(dataset_test)} samples')

datamodule = TBDataloader(dataset_train, dataset_val, dataset_test, batch_size=32)
print('Datasets and datamodule ready')

In [ ]:
# Inspect a sample to verify cell complex structure
sample = dataset_train[0]
if isinstance(sample, (list, tuple)) and len(sample) == 2:
    values, keys = sample
    print('Sample keys:', keys)
    for k, v in zip(keys, values):
        if hasattr(v, 'shape'):
            print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
        elif hasattr(v, 'size'):
            print(f'  {k}: size={v.size()}')
        else:
            print(f'  {k}: {type(v).__name__}')
else:
    print('Sample type:', type(sample))
    if hasattr(sample, 'keys'):
        for k in sample.keys():
            v = sample[k]
            if hasattr(v, 'shape'):
                print(f'  {k}: shape={v.shape}')
            else:
                print(f'  {k}: {type(v).__name__}')

## Backbone: CWN (CW Network)

CWN processes cell complexes using message passing across three ranks:
- **Rank 0** (nodes): receives messages from adjacent nodes and boundary of edges
- **Rank 1** (edges): receives messages from boundary nodes and co-boundary 2-cells
- **Rank 2** (2-cells / cycles): receives messages from boundary edges

The `CWNWrapper` handles the forward pass interface expected by `TBModel`,
feeding the cell complex connectivity matrices to the CWN backbone.

In [ ]:
# AllCellFeatureEncoder projects raw features at each rank to dim_hidden
# After CellCycleLifting + ProjectionSum, all ranks have `in_channels` features
feature_encoder = AllCellFeatureEncoder(
    in_channels=[in_channels] * n_cell_dims,
    out_channels=dim_hidden,
)

# CWN backbone from TopoModelX
backbone = CWN(
    in_channels_0=dim_hidden,
    in_channels_1=dim_hidden,
    in_channels_2=dim_hidden,
    hid_channels=dim_hidden,
    n_layers=4,
)

# CWNWrapper adapts the CWN interface for TBModel
backbone_wrapper = partial(
    CWNWrapper,
    out_channels=dim_hidden,
    num_cell_dimensions=n_cell_dims,
)

readout = PropagateSignalDown(**readout_config)
loss = TBLoss(**loss_config)
evaluator = TBEvaluator(**evaluator_config)
optimizer = TBOptimizer(**optimizer_config)

print('Components instantiated')
print(f'  CWN backbone: {backbone}')
print(f'  Feature encoder: {feature_encoder}')

In [ ]:
# Assemble the TBModel with CWN backbone + wrapper
model = TBModel(
    backbone=backbone,
    backbone_wrapper=backbone_wrapper,
    readout=readout,
    loss=loss,
    feature_encoder=feature_encoder,
    evaluator=evaluator,
    optimizer=optimizer,
    compile=False,
)

print(model)

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torchmetrics')

trainer = pl.Trainer(
    max_epochs=50,
    accelerator='cpu',
    enable_progress_bar=True,
    log_every_n_steps=1,
    enable_model_summary=False,
)
trainer.fit(model, datamodule)
train_metrics = trainer.callback_metrics

print('\nTraining finished. Collected metrics:')
for key, val in train_metrics.items():
    try:
        print(f'{key:25s} {float(val):.4f}')
    except Exception:
        print(key, val)

In [ ]:
trainer.test(model, datamodule)
test_metrics = trainer.callback_metrics
print('\nTest metrics:')
for key, val in test_metrics.items():
    try:
        print(f'{key:25s} {float(val):.4f}')
    except Exception:
        print(key, val)

## Notes

### CWN Architecture
CWN (CW Network) performs message passing on cell complexes, leveraging:
- `incidence_1` (nodes ↔ edges) and `incidence_2` (edges ↔ 2-cells)
- `adjacency_1` (node adjacency via edges)

The `CellCycleLifting` transform identifies cycles in the input graph and lifts them to 2-cells,
creating the cell complex structure that CWN requires.

### Key Differences from Custom Backbone Tutorial
- **Backbone**: Uses `topomodelx.nn.cell.cwn.CWN` instead of a custom `LightningModule`
- **Wrapper**: `CWNWrapper` handles the CWN-specific forward pass (feeding incidence/adjacency matrices)
- **Feature Encoder**: `AllCellFeatureEncoder` with `in_channels=[3, 3, 3]` encodes features at all 3 ranks
- **Readout**: `PropagateSignalDown` with `num_cell_dimensions=3` propagates signals from rank 2 → 1 → 0
- **Lifting**: `CellCycleLifting` is always applied (CWN requires cell complex data)

### Running via CLI
You can also run CWN through the TopoBench CLI:
```bash
python -m topobench model=cell/cwn dataset=graph/MUTAG
```